# 03 — EDA and Hypothesis Tests

All results are **observational / descriptive**, not causal. GDP and population are context variables, not treatments.

## Hypotheses tested
| ID | Hypothesis | Test | Metric |
|----|-----------|------|--------|
| H1 | Richer countries win more medals *per person* | Spearman ρ | GDP per capita × medals per million |
| H2 | More populous countries win more medals *in total* | Spearman ρ | population × total medals |
| H3 | Host countries outperform their own non-host baseline | Wilcoxon signed-rank (one-sided) | host rate vs typical non-host rate |
| H4 | Medal rate differs systematically across income quartiles | Kruskal-Wallis H | medals per million across Q1–Q4 |

**Why non-parametric tests?** `medals_per_million` is right-skewed with a few very high outliers (small, wealthy countries like Norway or New Zealand). Spearman and Kruskal-Wallis are rank-based and do not assume normality.  
**Why Wilcoxon for H3?** Each country is paired against itself — host vs. non-host editions — so a paired test is appropriate. One-sided alternative tests only the direction of interest (host > non-host).

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'tmp' / 'matplotlib'))

results = json.loads((ROOT / 'data/processed/analysis_results.json').read_text())

print('=== H1: GDP per capita vs medals per million ===')
h1 = results['h1_gdp_medals_per_million']
print(f"  Spearman ρ = {h1['spearman_rho']:.4f}  |  p = {h1['p_value']:.2e}")

print('\n=== H2: Population vs total medals ===')
h2 = results['h2_population_total_medals']
print(f"  Spearman ρ = {h2['spearman_rho']:.4f}  |  p = {h2['p_value']:.2e}")

print('\n=== H3: Host-country advantage ===')
h3 = results['h3_host_advantage']
print(f"  Paired countries: {h3['paired_countries']}")
print(f"  Median host rate:        {h3['median_host_rate']:.4f} medals/million")
print(f"  Median non-host rate:    {h3['median_typical_nonhost_rate']:.4f} medals/million")
if h3['p_value'] is not None:
    print(f"  Wilcoxon stat = {h3['wilcoxon_stat']:.1f}  |  p = {h3['p_value']:.4f}")

print('\n=== H4: Income quartile Kruskal-Wallis ===')
h4 = results['h4_income_quartiles']
print(f"  H = {h4['kruskal_stat']:.4f}  |  p = {h4['p_value']:.2e}")

print('\n=== Log-linear OLS model: log(medals) ~ log(GDP/cap) + log(pop) ===')
m = results['log_linear_model']
print(f"  n = {m['n']}  |  R² = {m['r_squared']:.3f}")
print(f"  Intercept = {m['intercept']:.3f}")
print(f"  β(log GDP/cap) = {m['log_gdp_beta']:.3f}")
print(f"  β(log pop)     = {m['log_population_beta']:.3f}")

## Visualisations

### Figure 1: Medal rate vs GDP per capita (scatter, host country flagged)

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(filename=str(ROOT / 'output/figures/gdp_medal_rate.png')))

**Reading the chart:** Each point is one country-Olympic-edition (e.g., USA at 2008 Summer). The x-axis is on a log scale. Red points are host editions. The upward trend is consistent with H1: wealthier countries tend to win more medals per person.

### Figure 2: Medal-rate distribution by income quartile (boxplot)

In [ ]:
ipy_display(Image(filename=str(ROOT / 'output/figures/income_quartile_medals.png')))

**Reading the chart:** Fliers are hidden; boxes show IQR. The monotone increase from Q1 to Q4 supports the Kruskal-Wallis result (H4 p ≈ 10⁻³⁰): income quartile is strongly associated with medal-rate distribution.

### Additional EDA: distribution of medals_per_million

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

data = pd.read_csv(ROOT / 'data/processed/olympics_wealth_country_edition.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(data.medals_per_million, bins=50, color='#2563eb', edgecolor='white')
axes[0].set_xlabel('Medals per million people')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of medal rate (raw)')

axes[1].hist(np.log1p(data.medals_per_million), bins=50, color='#16a34a', edgecolor='white')
axes[1].set_xlabel('log(1 + medals per million)')
axes[1].set_title('Distribution of medal rate (log-transformed)')
fig.suptitle('Right-skew justifies non-parametric tests', fontsize=12)
plt.tight_layout()
plt.show()

### Top 10 host editions by advantage gained

In [ ]:
host_editions = data[data.host_flag == 1][['Year','Season','NOC','medals','medals_per_million']].copy()
baselines = data[data.host_flag == 0].groupby('NOC')['medals_per_million'].median().rename('baseline_non_host')
host_editions = host_editions.merge(baselines, on='NOC', how='left')
host_editions['host_boost'] = host_editions['medals_per_million'] - host_editions['baseline_non_host']
print('Top 10 host editions by medal-rate boost over own baseline:')
display(host_editions.sort_values('host_boost', ascending=False).head(10).reset_index(drop=True))